# Local LangChain Vector + Graph Ingestion with Llama 3

## 本地模型

###LLM
使用 Ollama 和 llama3：

    ollama pull llama3.1

### 环境变量
 在 .env 文件中需要的变量，或在启动时加载为变量：

必需项：

    NEO4J_URI=...
    NEO4J_USERNAME=...
    NEO4J_PASSWORD=...

In [1]:
import os

CUSTOM_CACHE = r'F:\Teewon\Milvue\models'
os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')
os.environ['TORCH_HOME'] = CUSTOM_CACHE

In [2]:
from dotenv import load_dotenv
load_dotenv('../../.env')

True

In [3]:
from langchain_core.globals import set_verbose,set_debug
set_debug(True)
set_verbose(True)

### Milvus Lite Vectorstore

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_milvus import Milvus
from langchain_community.embeddings import HuggingFaceEmbeddings

urls=[
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs=[WebBaseLoader(url).load() for url in urls]
docs_list=[item for sublist in docs for item in sublist if item]
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=250,chunk_overlap=0)

doc_splits=text_splitter.split_documents(docs_list)

print(f"Number of docs: {len(docs_list)}")
print(f"Number of chunks: {len(doc_splits)}")

C:\Users\Administrator\AppData\Local\Temp\ipykernel_29328\503145139.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Number of docs: 3
Number of chunks: 187


In [5]:
# Add to Milvus
vectorstore=Milvus.from_documents(
    documents=doc_splits,
    collection_name="rag_milvus",
    embedding=HuggingFaceEmbeddings(),
    connection_args={"uri":"../../milvus_ingest.db"}
)
retriever=vectorstore.as_retriever()

C:\Users\Administrator\AppData\Local\Temp\ipykernel_29328\1880513296.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding=HuggingFaceEmbeddings(),
C:\Users\Administrator\AppData\Local\Temp\ipykernel_29328\1880513296.py:5: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding=HuggingFaceEmbeddings(),


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Neo4j Graphstore

In [8]:
from langchain_neo4j import Neo4jGraph
from langchain_ollama import ChatOllama
from langchain_experimental.graph_transformers import LLMGraphTransformer


# Initalize Neo4j
graph=Neo4jGraph()

# 图转换需要启用函数调用的LLM
# Graph Conversion requires function calling enabled llm
graph_llm=ChatOllama(model="llama3:latest",format="json")

# Filtered graph transformer
graph_transformer=LLMGraphTransformer(
    llm=graph_llm,
    allowed_nodes=["Person","Concept","Technology"],
    node_properties=["name","description","source"],
    allowed_relationships=["WROTE","MENTIONS","RELATED_TO"],
    strict_mode=False,
)

# Convert list of Document objects to Graph Document
graph_documents=graph_transformer.convert_to_graph_documents(doc_splits)

# Filter Graph Document with no nodes and relationships
filtered_graph_documents=[g_doc for g_doc in graph_documents if len(g_doc.nodes)>0 or len(g_doc.relationships)>0]

# Add Graph Documents to Neo4j
graph.add_graph_documents(filtered_graph_documents)

print(f"Graph documents pre-filter:{len(graph_documents)}, post-filter:{len(filtered_graph_documents)}")
print(f"1st Graph doc:{filtered_graph_documents[0].__dict__}")
print(f"Nodes from 1st graph doc:{filtered_graph_documents[0].nodes}")
print(f"Relationships from 1st graph doc:{filtered_graph_documents[0].relationships}")

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "LLM Powered Autonomous Agents | Lil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n|\n\n\n\n\n\n\nPosts\n\n\n\n\nArchive\n\n\n\n\nSearch\n\n\n\n\nTags\n\n\n\n\nFAQ\n\n\n\n\n\n\n\n\n\n      LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\n \n\n\nTable of Contents\n\n\n\nAgent System Overview\n\nComponent One: Planning\n\nTask Decomposition\n\nSelf-Reflection\n\n\nComponent Two: Memory\n\nTypes of Memory\n\nMaximum Inner Product Search (MIPS)\n\n\nComponent Three: Tool Use\n\nCase Studies\n\nScientific Discovery Agent\n\nGenerative Agents Simulation\n\nProof-of-Concept Examples\n\n\nChallenges\n\nCitation\n\nReferences"
}
[chain/start] [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "input": "LLM Powered Autonomous Age